In [ ]:
# Imports
import os
import pandas as pd
from pathlib import Path
import statistics
import re
import nltk
import string
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from itertools import chain
from collections import Counter
import unicodedata
import numpy as np
from nltk.tokenize import word_tokenize
from nltk import FreqDist, ngrams, pos_tag
import statsmodels.api as sm
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from itertools import chain

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
# Files
project_dir = Path.cwd()
main_dir = project_dir / 'dataverse' / 'dataverse' / 'UN General Debate Corpus' / 'TXT'

txt_files = []
for root, dirs, files in os.walk(main_dir):
    for file in files:
        if file.endswith('.txt'):
            txt_files.append(Path(root) / file)


In [ ]:
# Getting the texts 

texts = {}

for path in txt_files:
    with open(path, 'r', encoding='utf-8') as f:
        texts[path] = f.read()

In [ ]:
# Dataframe 

records = []

for path, text in texts.items():
    p = Path(path)
    stem = p.stem              
    parts = stem.split("_")

    if len(parts) != 3:
        print("Wrong filename, continue:", p.name)
        continue

    iso, nr, year = parts

    if not nr.isdigit() or not year.isdigit():
        print("Not int, continue", p.name)
        continue

    records.append({
        "iso": iso,
        "session_nr": int(nr),
        "year": int(year),
        "path": path,
        "text": text
    })

df = pd.DataFrame(records).sort_values("year").reset_index(drop=True)

# Text Preprocessing

In [ ]:
len(df['iso'].unique())

In [ ]:
df_example = df[:5]
df_example

In [ ]:
# Step 1: Parsing and Tokenization

df['text_clean'] = df['text'].copy()

# Remove newline characters and extra spaces
df['text_clean'] = df['text_clean'].apply(lambda x: re.sub(r'\s+', ' ', x))

# Tokenize the texts
df['tokens'] = df['text_clean'].apply(nltk.word_tokenize)


In [ ]:
# Step 2: Cleaning Punctuation and Converting to Lowercase

# Normalization
df['tokens'] = df['tokens'].apply(
    lambda lst: [
        w.replace('``', '')      
         .replace("''", '')      
         .replace("“", '')       
         .replace("”", '')
         .replace("‘", '')
         .replace("’", '')
         .replace(".", '')
         .replace("-", '')
        for w in lst
    ]
)


# Empty strings
df['tokens'] = df['tokens'].apply(lambda lst: [w for w in lst if w not in ['', ' ']])

# Punctuation
punct = set(string.punctuation)
df['tokens'] = df['tokens'].apply(lambda lst: [w for w in lst if w not in punct])

# Initialize stopwords and add custom stopwords
stop_words = set(stopwords.words('english'))
custom_stopwords = {
    'mr','mrs','madam','president','chairman','chairwoman','chair',
    'secretarygeneral','secretary-general','excellency','excellencies',
    'distinguished','delegates','delegate','ladies','gentlemen','assembly',
    'general','session','committee','thank','thanks', 'also', 'must', 'would', 'one', 'us',
    'region','problem','support','effort',

    'international','community','global','world','people','country','countries',
    'state','states','government','governments','united','nations','organization',

    'honour','honor','privilege','pleased','allow','permit','begin','address',

    'un','unga','security','council','generalassembly','agenda','item',

    'representative','delegation','minister','prime','minister','ambassador'
}  
stop_words.update(custom_stopwords)
df['tokens'] = df['tokens'].apply(lambda lst: [w for w in lst if w.lower() not in stop_words])

# Initialize stemmer
stemmer = PorterStemmer()
df['tokens'] = df['tokens'].apply(lambda lst: [stemmer.stem(w) for w in lst])


df['clean_text'] = df['tokens'].apply(lambda lst: " ".join(lst))
df = df[['iso', 'session_nr', 'year', 'text', 'clean_text', 'tokens']]

In [ ]:
# Removing speakers from tokens
speakers = pd.read_excel(project_dir / 'dataverse' / 'dataverse' / 'Speakers_by_session.xlsx')
speaker_names = speakers["Name of Person Speaking"].dropna().tolist()

name_tokens = set()

for full_name in speaker_names:
    parts = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", full_name)  
    for p in parts:
        name_tokens.add(p.lower())

df["tokens"] = df["tokens"].apply(lambda lst: [w for w in lst if w not in name_tokens])


In [ ]:
#df.to_pickle("ungdc_cleaned.pickle")

# Exploratory Analysis

In [ ]:
# Basic descriptive statistics of the texts length

texts_cleaned = df['clean_text']
texts_lengths = [len(text) for text in texts_cleaned]  

print("Min. =", np.round(min(texts_lengths),2))                     
print("Max. =", np.round(max(texts_lengths),2))                     
print("Mean =", np.round(statistics.mean(texts_lengths),2))         
print("Median =", np.round(statistics.median(texts_lengths),2))     
print("Std. dev. =", np.round(statistics.stdev(texts_lengths),2))   


In [ ]:
# Histogram of text lengths
plt.hist(texts_lengths, bins=100)
plt.xlabel('Text length')
plt.ylabel('Number of texts')
plt.show()

In [ ]:
# Step 3: Creating Term Frequency Matrix (All texts)

all_tokens = list(chain.from_iterable(df['tokens']))

# Compute term frequencies
tf_all = Counter(all_tokens)
tf_all_df = (
    pd.DataFrame.from_dict(tf_all, orient='index', columns=['frequency'])
      .sort_values(by='frequency', ascending=False)
)
print("Global Term Frequency (Top 20):")
print(tf_all_df.head(20))

# Trade related words
trade_target = {'trade', 'import', 'export','tariff', 'wto', 'sanction', 'embargo', 'globalis', 'liberalis', 'protect', 'supply', 'logistic'}
tf_trade = {w: tf_all[w] for w in trade_target}
print("\nTrade-related term frequencies:")
print(tf_trade)

In [ ]:
# Step 4: Computing Top 10 Word Co-occurrences (Simple Counts) within a Context Window

# context window size 
window_size = 5

# Function to generate co-occurrences within a context window
def generate_cooccurrences(token_lists, window_size):
    cooccurrence_counts = Counter()
    for tokens in token_lists:
        for idx, word in enumerate(tokens):
            
            start = max(0, idx - window_size)
            end = min(len(tokens), idx + window_size + 1)

            context_words = tokens[start:idx] + tokens[idx+1:end]
            for context_word in context_words:
                if word != context_word:

                    pair = frozenset([word, context_word])
                    cooccurrence_counts[pair] += 1
    return cooccurrence_counts

cooccurrence_counts_all = generate_cooccurrences(df['tokens'], window_size)


top_10_cooccurrences = cooccurrence_counts_all.most_common(10)

print("\nTop 10 Word Co-occurrences for Entire Corpus (Simple Counts):")
for pair, count in top_10_cooccurrences:
    print(f"{set(pair)}: {count}")


In [ ]:
# Target - Trade related co-occurrences

def cooccur_target(token_lists, target_words, window_size=5):
    co_counts = Counter()
    
    for tokens in token_lists:
        for idx, word in enumerate(tokens):
            if word in target_words:
                start = max(0, idx - window_size)
                end = min(len(tokens), idx + window_size + 1)

                context = tokens[start:idx] + tokens[idx+1:end]
                for c in context:
                    if c not in target_words:
                        pair = (word, c)  
                        co_counts[pair] += 1
    
    return co_counts

target = {'trade', 'import', 'export','tariff', 'wto', 'sanction', 'embargo', 'globalis', 'liberalis', 'protect', 'supply', 'logistic'}

co_trade = cooccur_target(df['tokens'], target_words=target, window_size=5)

# Top co-occurrences
top20_trade = co_trade.most_common(10)

for pair, count in top20_trade:
    print(pair, count)


In [ ]:
# Step 5: Computing PMI for Co-occurrences  (co-occurence in the window)
# Only target words

target_words = {"trade", "import", "export"}
window_size = 5
years_to_analyze = [2000, 2010, 2024]

def cooc_for_targets(token_lists, target_words, window_size=5):
    co_counts = Counter()
    for tokens in token_lists:
        for i, w in enumerate(tokens):
            if w in target_words:
                start = max(0, i - window_size)
                end = min(len(tokens), i + window_size + 1)
                context = tokens[start:i] + tokens[i+1:end]
                for c in context:
                    if c not in target_words:  
                        pair = (w, c)   
                        co_counts[pair] += 1
    return co_counts

for yr in years_to_analyze:
    print(f"\n===== PMI for TARGET WORDS in {yr} =====")

    subset = df[df['year'] == yr]

    if subset.empty:
        print("No speeches.")
        continue
    
    all_words = list(chain.from_iterable(subset['tokens']))
    total_words = len(all_words)
    word_freq = Counter(all_words)

    # co-occurrence
    co_counts = cooc_for_targets(subset['tokens'], target_words, window_size)

    # PMI
    pmi_scores = {}
    total_co = sum(co_counts.values())

    for (w1, w2), co_count in co_counts.items():

        if co_count < 2:
            continue
        if word_freq[w2] < 3:
            continue

        p_w1 = word_freq[w1] / total_words
        p_w2 = word_freq[w2] / total_words
        p_w1_w2 = co_count / total_co

        pmi = np.log2(p_w1_w2 / (p_w1 * p_w2))
        pmi_scores[(w1, w2)] = pmi

    sorted_pmi = sorted(pmi_scores.items(), key=lambda x: x[1], reverse=True)

    print("Top 10 PMI pairs:")
    for (w1, w2), score in sorted_pmi[:10]:
        print(f"{w1} – {w2}: {score:.4f}")

In [ ]:
# Word Cloud (trade speeches)

trade_words = {'trade', 'import', 'export'}  
threshold = 5  

def count_trade_tokens(tokens):
    return sum(1 for w in tokens if w in trade_words)

df['trade_count'] = df['tokens'].apply(count_trade_tokens)

# only for trade speeches
trade_heavy = df[df['trade_count'] >= threshold]

print(f"Number of trade-heavy speeches: {len(trade_heavy)}")

text_trade = " ".join(trade_heavy['clean_text'])

wc = WordCloud(
    width=1400,
    height=700,
    background_color='white',
    stopwords=STOPWORDS,
    colormap='viridis'
).generate(text_trade)

plt.figure(figsize=(15, 8))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title(f"Word Cloud – trade-heavy speeches (≥{threshold} words trade/import/export)", fontsize=18)
plt.show()


# Trade analysis

In [ ]:
#df = pd.read_pickle("ungdc_cleaned.pickle")

In [ ]:
# Filtering tokens only to trade related 

trade_dict = {
    "trade", "import", "export", "tariff", "quota", "custom", "dut",
    "market", "border", "access", "liber", "protect", "competit", 
    "subsid", "dump", "sanct", "embargo", "bilater", "multilater",
    "agreement", "disput", "nego", "cooper", "global", "integr",
    "flow", "invest", "product", "commerce", "ecommerce",

    "wto", "gatt", "unctad", "imf", "worldbank", "fta", 
    "partnership", "bloc", "union",

    "sanction", "embargo", "restrict", "blockad", "penalti", 
    "disrupt", "war", "tension", "measure",

    "oil", "gas", "energi", "energ", "commod", "grain", "food", "resourc", 
    "raw", "mineral", "supply",

    "chain", "logist", "transport", "ship", "port", "rout", "link",
    "interdepend", "resilienc", "shortag",

    "fdi", "invest", "capit", "industr", "manufactur", "firm", 
    "compani", "sector",

    "resh", "offsh", "shore", "reshore", "offshore"
}

df["trade_tokens"] = df["tokens"].apply(
    lambda lst: [w for w in lst if w in trade_dict]
)

In [ ]:
# Trade related sentences

# Dividing to sentences
df["sentences"] = df["text"].apply(lambda t: nltk.sent_tokenize(t))

# Stemming
stemmer = PorterStemmer()

def is_trade_sentence(sentence, trade_dict):
    # tokenization
    tokens = word_tokenize(sentence.lower())

    # stemming
    stems = [stemmer.stem(w) for w in tokens]

    return any(stem in trade_dict for stem in stems)

df["trade_sentences"] = df["sentences"].apply(
    lambda sent_list: [s for s in sent_list if is_trade_sentence(s, trade_dict)]
)

In [ ]:
# Number of trade sentences
df["n_trade_sentences"] = df["trade_sentences"].apply(len)

# Percent of trade sentences
df["trade_sentence_ratio"] = df["n_trade_sentences"] / df["sentences"].apply(len)

# Overall trade share
df["trade_sentence_ratio"].mean()

In [ ]:
# Aggregate by year
yearly_trade = df.groupby("year")["trade_sentence_ratio"].mean().reset_index()

plt.figure(figsize=(14, 7))
plt.plot(
    yearly_trade["year"],
    yearly_trade["trade_sentence_ratio"],
    linewidth=2.5
)

plt.title("Average Share of Trade-Related Sentences per Year, 1946–2024", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Trade Sentence Share", fontsize=14)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Sentiment

## Trade focus by country and decade


In [ ]:
# Trade focus heatmap by decade and country
# This chart compares the countries with the highest average share of trade-related sentences.
country_trade = (
    df.groupby('iso')
    .agg(
        avg_trade_sentence_ratio=('trade_sentence_ratio', 'mean'),
        n_speeches=('trade_sentence_ratio', 'size')
    )
    .query('n_speeches >= 10')
    .sort_values('avg_trade_sentence_ratio', ascending=False)
)

top_countries = country_trade.head(15).index
df_top_trade = df[df['iso'].isin(top_countries)].copy()
df_top_trade['decade'] = (df_top_trade['year'] // 10) * 10

trade_heatmap = (
    df_top_trade
    .groupby(['iso', 'decade'])['trade_sentence_ratio']
    .mean()
    .unstack('decade')
    .reindex(top_countries)
)

plt.figure(figsize=(14, 8))
plt.imshow(trade_heatmap, aspect='auto', cmap='YlGnBu')
plt.colorbar(label='Average share of trade-related sentences')
plt.xticks(range(len(trade_heatmap.columns)), trade_heatmap.columns, rotation=45)
plt.yticks(range(len(trade_heatmap.index)), trade_heatmap.index)
plt.title('Trade-related sentence share by decade for the most trade-focused countries')
plt.xlabel('Decade')
plt.ylabel('Country ISO code')
plt.tight_layout()
plt.show()


In [ ]:
import nltk
import os
import ssl
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', download_dir='/Users/weronika/nltk_data')


In [ ]:
# Sentiment of trade sentences

sia = SentimentIntensityAnalyzer()

def sentiment_stats_for_sentences(sent_list, analyzer, pos_th=0.05, neg_th=-0.05):
    """
    sent_list: list of sentences for one speech
    pos_th, neg_th: treshorlds 
    """
    if not sent_list:
        return {
            "sent_mean": None,
            "sent_pos_share": None,
            "sent_neg_share": None,
            "sent_neu_share": None
        }

    scores = []
    labels = []

    for s in sent_list:
        sc = analyzer.polarity_scores(s)["compound"]
        scores.append(sc)
        if sc >= pos_th:
            labels.append("pos")
        elif sc <= neg_th:
            labels.append("neg")
        else:
            labels.append("neu")

    n = len(scores)
    sent_mean = sum(scores) / n

    sent_pos_share = labels.count("pos") / n
    sent_neg_share = labels.count("neg") / n
    sent_neu_share = labels.count("neu") / n

    return {
        "sent_mean": sent_mean,
        "sent_pos_share": sent_pos_share,
        "sent_neg_share": sent_neg_share,
        "sent_neu_share": sent_neu_share
    }


In [ ]:
sent_polarity = df['trade_sentences'].apply(
    lambda lst: sentiment_stats_for_sentences(lst, sia)
)

# Add polarity stats to df
df["trade_sent_mean"]      = sent_polarity.apply(lambda d: d["sent_mean"]) 
df["trade_sent_pos_share"] = sent_polarity.apply(lambda d: d["sent_pos_share"])
df["trade_sent_neg_share"] = sent_polarity.apply(lambda d: d["sent_neg_share"])
df["trade_sent_neu_share"] = sent_polarity.apply(lambda d: d["sent_neu_share"])

In [ ]:
# Chart

df_sent = df[df["trade_sent_mean"].notna()]

yearly_sent = (
    df_sent
    .groupby("year")["trade_sent_mean"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 7))

plt.plot(
    yearly_sent["year"],
    yearly_sent["trade_sent_mean"],
    linewidth=2
)

# linia neutralnego sentymentu (0)
plt.axhline(0, linestyle="--")

plt.title("Average sentiment score of trade-related sentences, 1946–2024", fontsize=15)
plt.xlabel("Year", fontsize=13)
plt.ylabel("Average sentiment (VADER compound)", fontsize=13)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## N-gram analysis

In [ ]:
# Unigrams 

# All tokens about trade
all_trade_tokens = [t for lst in df["trade_tokens"] for t in lst]

# Unigrams
unigrams = FreqDist(all_trade_tokens)
print("Most Common Unigrams:")
print(unigrams.most_common(20))

In [ ]:
# Bigrams

def bigrams_for_speech(tokens):
    return list(ngrams(tokens, 2))

df["trade_bigrams"] = df["trade_tokens"].apply(bigrams_for_speech)
all_bigrams = [bg for lst in df["trade_bigrams"] for bg in lst]

# Adding filter to avoid repetition
all_bigrams = [
    tuple(sorted(bg))
    for lst in df["trade_bigrams"]
    for bg in lst
    if bg[0] != bg[1]
]

FreqDist(all_bigrams).most_common(20)


In [ ]:
# Trigrams

def trigrams_for_speech(tokens):
    return list(ngrams(tokens, 3))

df["trade_trigrams"] = df["trade_tokens"].apply(trigrams_for_speech)

all_trigrams = [tg for lst in df["trade_trigrams"] for tg in lst]

# Filter (avoid any repetition)
all_trigrams = [
    tuple(sorted(tg))
    for lst in df["trade_trigrams"]
    for tg in lst
    if len(set(tg)) == 3
]

trigram_freq = FreqDist(all_trigrams)
trigram_freq.most_common(20)


# TF-IDF matrix for trade words

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Sorting the list of trade words
trade_vocab = sorted(list(trade_dict))

vectorizer = TfidfVectorizer(
    vocabulary=trade_vocab,   
    tokenizer=str.split,      
    preprocessor=None,
    lowercase=False           
)

tfidf_matrix = vectorizer.fit_transform(df["clean_text"])

In [ ]:
# Converting to dataframe

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=trade_vocab,
    index=df.index
)

# Adding columns with country id and year
tfidf_df["iso"] = df["iso"]
tfidf_df["year"] = df["year"]

In [ ]:
tfidf_df

In [ ]:
# "Trade intensity" indicator
# The sum of all columns per speech

trade_cols = trade_vocab  
tfidf_df["trade_intensity_tfidf"] = tfidf_df[trade_cols].sum(axis=1)

# The most trade-intensive speeches
tfidf_df.sort_values("trade_intensity_tfidf", ascending=False).head(10)[["iso","year","trade_intensity_tfidf"]]


## Liberal vs Protection 

In [ ]:
liberal_words = {
    "liber", "multilater", "cooper", "facilit", "integr", "open",
    "market", "access", "stabil", "reform", "support", "develop",
    "partnership", "connect", "dialogu", "agreement","agree" ,"nego",
    "resilienc", "interdepend"
}

protection_words = {
    "protect", "restrict", "sanction", "tariff", "duti", "quota",
    "block", "embargo", "penalti", "retali", "impos", "barrier",
    "subsidi", "dump", "war", "tension", "shortag", "controversi",
    "disrupt", "weapon"
}

# Liberalization score (tfidf sum for liberal words)
liberal_cols = [w for w in trade_vocab if w in liberal_words]
tfidf_df["liberal_score"] = tfidf_df[liberal_cols].sum(axis=1)

# Protectionism score
protection_cols = [w for w in trade_vocab if w in protection_words]
tfidf_df["protection_score"] = tfidf_df[protection_cols].sum(axis=1)

# Index: protection - liberal (for exploration)
tfidf_df["protection_minus_liberal"] = (tfidf_df["protection_score"] - tfidf_df["liberal_score"])


In [ ]:
# Narration trend by year (for chart)
trend = (tfidf_df.groupby("year")[["liberal_score", "protection_score", "protection_minus_liberal"]].mean().reset_index())
trend

In [ ]:
def word_ratio(tokens, keywords):
    return sum(1 for t in tokens if any(t.startswith(k) for k in keywords)) / len(tokens) if len(tokens) > 0 else 0

df["lib_ratio"]  = df["tokens"].apply(lambda x: word_ratio(x, liberal_words))
df["prot_ratio"] = df["tokens"].apply(lambda x: word_ratio(x, protection_words))

ratio_year = df.groupby("year")[["lib_ratio", "prot_ratio"]].mean().reset_index()


In [ ]:
plt.figure(figsize=(12,6))

plt.plot(ratio_year["year"], ratio_year["lib_ratio"], linewidth=2, label="Liberalization", color="green")
plt.plot(ratio_year["year"], ratio_year["prot_ratio"], linewidth=2, label="Protectionism", color="red")

plt.title("Liberalization vs Protectionism in Trade Narratives (1946–2024)")
plt.xlabel("Year")
plt.ylabel("Share of Keywords")
plt.legend()
plt.grid(alpha=0.3)

plt.show()


# Topic Modelling
## LDA

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import math

In [ ]:
df["trade_only_text"] = df["trade_tokens"].apply(lambda lst: " ".join(lst))

vectorizer = TfidfVectorizer(
    vocabulary=trade_vocab,
    tokenizer=str.split,
    preprocessor=None,
    lowercase=False  
)

text_matrix = vectorizer.fit_transform(df["trade_only_text"].fillna(""))


In [ ]:
# Sanity check - proper number of topics

ks = [4, 5, 6, 7, 8]
results = []

for k in ks:
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method='batch',
        max_iter=20
    )
    lda.fit(text_matrix)
    perplexity = lda.perplexity(text_matrix)
    results.append((k, perplexity))

results

# I will select the number of topics based on perplexity (lowest) and interpretability
# Models with 4 and 5 topics performs best


In [ ]:
# LDA 

lda = LatentDirichletAllocation(
    n_components=5,   # tested with 4 and 5
    random_state=42,
    learning_method='batch',
    max_iter=20
)

lda.fit(text_matrix)

In [ ]:
# 5 topics
def display_topics(model, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx}:")
        print(", ".join([feature_names[i]
                        for i in topic.argsort()[:-n_top_words - 1:-1]]))

tfidf_feature_names = vectorizer.get_feature_names_out()
display_topics(lda, tfidf_feature_names)

# Decided to go for model with 5 topics since the interpretability seems better

In [ ]:
# Share of topics in every speech

topic_distribution = lda.transform(text_matrix)

for i in range(lda.n_components):
    df[f"topic_{i}_share"] = topic_distribution[:, i]

# Share of topics per years
topic_cols = [f"topic_{i}_share" for i in range(5)]
topic_trends = (
    df.groupby("year")[topic_cols]
      .mean()
      .reset_index()
)



In [ ]:
# Topics Chart (setting numering from 1)

top_n = 10
n_topics = lda.n_components  # number of topics discovered by LDA

n_cols = 2
n_rows = math.ceil(n_topics / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for idx, component in enumerate(lda.components_):

    zipped = list(zip(tfidf_feature_names, component))
    top_terms = sorted(zipped, key=lambda t: t[1], reverse=True)[:top_n]
    words, weights = zip(*top_terms)

    ax = axes[idx]

    ax.barh(words[::-1], weights[::-1], color='steelblue')

    ax.set_title(f"Topic {idx + 1}", fontsize=14)

    ax.set_xlabel("Weight", fontsize=12)
    ax.set_ylabel("Token", fontsize=12)
    ax.tick_params(axis='both', labelsize=11)

for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()



In [ ]:
# Time series chart

plt.figure(figsize=(14,7))

colors = [
    "#1f78b4", 
    "#33a02c", 
    "#e31a1c",
    "#ff7f00", 
    "#6a3d9a"  
]

for i, (col, c) in enumerate(zip(topic_cols, colors)):
    
    label_name = f"Topic {i + 1}"
    
    plt.plot(
        topic_trends["year"],
        topic_trends[col],
        label=label_name,
        linewidth=3.0,
        color=c
    )

plt.title("Topic Share Over Time (Smoothed, LDA Trade Narratives 1946–2024)",
          fontsize=16)
plt.xlabel("Year", fontsize=13)
plt.ylabel("Average Topic Share", fontsize=13)
plt.grid(alpha=0.3)
plt.legend(title="Topics", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Stacked chart

plt.figure(figsize=(14,7))

labels = [f"Topic {i+1}" for i in range(len(topic_cols))]

plt.stackplot(
    topic_trends["year"],
    [topic_trends[col] for col in topic_cols],
    labels=labels,
    colors=colors,
    alpha=0.85
)

plt.title("Stacked Area Chart – Trade Narratives Over Time (1946–2024)",
          fontsize=16)
plt.xlabel("Year", fontsize=13)
plt.ylabel("Average Topic Share", fontsize=13)
plt.legend(title="Topics", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=11)

plt.tight_layout()
plt.show()


# Event analysis

In [ ]:
# WTO

def keyword_ratio(tokens, keywords):
    return sum(1 for t in tokens if any(t.startswith(k) for k in keywords)) / len(tokens) if len(tokens) > 0 else 0

wto_words = ["wto", "gatt", "uruguay", "doha",
 "accession", "accession", "member", 
 "rules-based", "multilater", "liberaliz",
 "negotiat", "trade dispute", "tariff barrier",
 "trade round", "uruguay round", "doha round"]
df["wto_ratio"] = df["tokens"].apply(lambda x: keyword_ratio(x, wto_words))
wto_year = df.groupby("year")["wto_ratio"].mean().reset_index()

lowess = sm.nonparametric.lowess(wto_year["wto_ratio"], wto_year["year"], frac=0.15)
plt.plot(lowess[:,0], lowess[:,1], color="black")

plt.figure(figsize=(12,6))
plt.plot(wto_year["year"], wto_year["wto_ratio"], linewidth=2)

plt.axvline(1995, color='red', linestyle='--', alpha=0.7, label="WTO creation (1995)")

plt.title("WTO-related Keywords Over Time")
plt.xlabel("Year")
plt.ylabel("Share of WTO Keywords")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
# Financial Crisis

crisis_words = ["financ", "crisis", "credit", "liquid", "market", "instabil",
 "bank", "recession", "shock", "downturn",
 "stimulus", "recover", "regulat", "oversight",
 "imf", "bailout"]
df["crisis_ratio"] = df["tokens"].apply(lambda x: keyword_ratio(x, crisis_words))
crisis_year = df.groupby("year")["crisis_ratio"].mean().reset_index()

lowess = sm.nonparametric.lowess(crisis_year["crisis_ratio"], crisis_year["year"], frac=0.15)
plt.plot(lowess[:,0], lowess[:,1], color="black")

plt.figure(figsize=(12,6))
plt.plot(crisis_year["year"], crisis_year["crisis_ratio"], color="darkred", linewidth=2)

plt.axvline(2008, color='black', linestyle='--', alpha=0.7, label="Global Financial Crisis (2008)")

plt.title("Protectionist / Crisis Keywords Over Time")
plt.xlabel("Year")
plt.ylabel("Share of Protectionist Keywords")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
# COVID

covid_words = ["covid", "corona", "virus", "pandem", "epidem",
    "vaccine", "vaccin", "immun", "lockdown",
    "quarantin",  "supply", "chain", "disrupt", "shortag", 
    "resilienc", "stockpil", "bottleneck",  "shutdown", "remote", "distanc",  
    "reopen", "stimulus", "recovery"]
df["covid_ratio"] = df["tokens"].apply(lambda x: keyword_ratio(x, covid_words))
covid_year = df.groupby("year")["covid_ratio"].mean().reset_index()

plt.figure(figsize=(12,6))
plt.plot(covid_year["year"], covid_year["covid_ratio"], color="purple", linewidth=2)

plt.axvline(2020, color='black', linestyle='--', alpha=0.7, label="COVID-19 (2020)")

plt.title("Supply Chain / Resilience Keywords Over Time")
plt.xlabel("Year")
plt.ylabel("Share of COVID-related Keywords")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
# Trump

trump_words = [
    "erosion", "retreat", "withdraw", "withdrawal",
    "unilateral", "isolation", "fragment", "nationalism",
    "populism", "protectionism", "sovereigntist",  "rules-based", "rules", "order", "undermin", "threat","tariff", "retaliat", "sanction", "pressure",
    "escalat", "coerc", "compete", "rivalry", 
    "great-power", "competition", "confront","indo-pacific", "asia-pacific", "strategic", "power",
]
df["trump_ratio"] = df["tokens"].apply(lambda x: keyword_ratio(x, trump_words))
trump_year = df.groupby("year")["trump_ratio"].mean().reset_index()

plt.figure(figsize=(12,6))
plt.plot(trump_year["year"], trump_year["trump_ratio"], color="orange", linewidth=2)

plt.axvline(2017, color='black', linestyle='--', alpha=0.7, label="Trump start (2017)")
plt.axvline(2020, color='black', linestyle='--', alpha=0.4, label="Trump end (2020)")

plt.title("Trade War / Sovereignty Keywords Over Time")
plt.xlabel("Year")
plt.ylabel("Share of Keywords")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
